# Aplikasi Pengenalan Wajah Mahasiswa (Real-Time)

Aplikasi ini dibuat menggunakan library `face_recognition` (berbasis `dlib`) dan `OpenCV` (`cv2`) untuk mendeteksi serta mengenali wajah mahasiswa secara *real-time* melalui kamera/webcam. Referensi wajah diambil secara otomatis dari file gambar yang berada di folder `dataset/`.

### Petunjuk Setup Kernel di VSCode:
1. Buka notebook ini (`main.ipynb`) di VSCode.
2. Di pojok kanan atas notebook, klik tombol **"Select Kernel"**.
3. Pilih **Python Environments...**.
4. Pilih virtual environment yang sudah kita buat, yaitu `./venv/bin/python` (atau kernel bertuliskan `venv (Python 3.14.x)`).
5. Setelah kernel terpilih, Anda siap menjalankan cell-cell di bawah ini.

---  
### Langkah 1: Import Library yang Diperlukan
Kita akan menggunakan OpenCV untuk mengakses webcam, `face_recognition` untuk ekstraksi wajah, `numpy` untuk operasi array matematika cepat, serta `os` dan `glob` untuk manajemen file dataset.

In [ ]:
import os
import glob
import cv2
import face_recognition
import numpy as np

print("Pustaka berhasil di-import!")
print("OpenCV Version:", cv2.__version__)

---  
### Langkah 2: Memuat Dataset dan Ekstraksi Wajah (Face Encodings)
Cell ini akan membaca semua foto di dalam folder `dataset/`. Nama mahasiswa diambil dari nama file foto tersebut (misal: file `Abdi (1).JPG` akan didaftarkan sebagai nama **Abdi**). Wajah dari setiap foto akan dipindai dan dikonversi menjadi representasi numerik (encoding 128-dimensi) yang unik untuk proses perbandingan nantinya.

In [ ]:
# Path ke folder dataset
dataset_path = "dataset"

# List untuk menyimpan representasi wajah (encoding) dan nama mahasiswa
known_face_encodings = []
known_face_names = []

# Mencari semua file foto dengan ekstensi umum
image_extensions = ["*.JPG", "*.jpeg", "*.png", "*.jpg", "*.PNG", "*.JPEG"]
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(dataset_path, ext)))

print(f"Menemukan {len(image_paths)} file gambar di folder '{dataset_path}'.")
print("Memulai ekstraksi wajah (face encodings)... Mohon tunggu sebentar.")

for path in image_paths:
    filename = os.path.basename(path)
    # Mengambil nama file tanpa ekstensi
    name = os.path.splitext(filename)[0]
    
    # Bersihkan nama jika mengandung nomor/kurung, misal "Abdi (1)" menjadi "Abdi"
    if " (" in name:
        name = name.split(" (")[0]
        
    try:
        # Membaca file gambar
        image = face_recognition.load_image_file(path)
        
        # Ekstrak encoding wajah (mengambil wajah pertama yang ditemukan)
        face_encodings = face_recognition.face_encodings(image)
        
        if len(face_encodings) > 0:
            known_face_encodings.append(face_encodings[0])
            known_face_names.append(name)
            print(f"[Sukses] Terdaftar: {name} ({filename})")
        else:
            print(f"[Peringatan] Tidak mendeteksi wajah di file: {filename}")
            
    except Exception as e:
        print(f"[Error] Gagal membaca {filename}: {str(e)}")

print("\n--- Proses Selesai ---")
print(f"Berhasil mendaftarkan {len(known_face_encodings)} mahasiswa ke dalam sistem.")

---  
### Langkah 3: Pengenalan Wajah Real-Time Menggunakan Kamera
Cell ini akan mengaktifkan kamera laptop/webcam eksternal Anda. 

**Cara kerja:**
1. Mengambil frame video secara *real-time*.
2. Memperkecil frame (resize ke 0.25) untuk optimalisasi performa agar tidak lag di CPU.
3. Mendeteksi lokasi wajah di dalam frame kamera dan menghitung encoding-nya.
4. Membandingkan dengan encoding database mahasiswa.
5. Menampilkan bounding box warna **Hijau** dengan nama mahasiswa jika cocok, atau **Merah** dengan tulisan "Tidak Dikenal" jika tidak cocok.
6. **Tekan tombol 'q'** di keyboard (saat jendela kamera aktif) untuk menghentikan program dan menutup kamera.

In [ ]:
# Inisialisasi kamera bawaan (index 0)
video_capture = cv2.VideoCapture(0)

if not video_capture.isOpened():
    print("Error: Kamera tidak dapat diakses. Pastikan izin kamera sudah diberikan atau tidak dipakai aplikasi lain.")
else: 
    print("Kamera aktif! Jendela video eksternal akan segera muncul.")
    print("PENTING: Tekan tombol 'q' pada keyboard di jendela video untuk menutup kamera.")

    while True:
        # Baca frame dari kamera
        ret, frame = video_capture.read()
        if not ret:
            print("Gagal membaca frame kamera.")
            break

        # Resize frame menjadi 1/4 ukuran asli agar deteksi berjalan lancar (real-time FPS tinggi)
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)

        # OpenCV menggunakan BGR, sedangkan face_recognition membutuhkan RGB
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        # Deteksi semua lokasi wajah dan ekstraksi encoding-nya di frame aktif
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        face_names = []
        for face_encoding in face_encodings:
            # Bandingkan wajah dengan database mahasiswa
            # Tolerance=0.5 adalah tingkat toleransi jarak. Lebih rendah = lebih ketat/akurat.
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=0.5)
            name = "Tidak Dikenal"

            # Cari jarak kecocokan wajah terkecil
            face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
            if len(face_distances) > 0:
                best_match_index = np.argmin(face_distances)
                if matches[best_match_index]:
                    name = known_face_names[best_match_index]

            face_names.append(name)

        # Menampilkan gambar kotak pembatas dan nama mahasiswa di atas frame asli
        for (top, right, bottom, left), name in zip(face_locations, face_names):
            # Kembalikan koordinat wajah karena sebelumnya diperkecil ke 1/4 (dikalikan 4)
            top *= 4
            right *= 4
            bottom *= 4
            left *= 4
            
            # Tentukan warna kotak: Hijau jika dikenal, Merah jika Tidak Dikenal
            color = (0, 255, 0) if name != "Tidak Dikenal" else (0, 0, 255)

            # Gambar kotak merah/hijau di sekitar wajah
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

            # Tulis background label nama di bawah wajah
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            font = cv2.FONT_HERSHEY_DUPLEX
            cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.7, (255, 255, 255), 1)

        # Tampilkan frame hasil akhir ke jendela pop-up OpenCV
        cv2.imshow('Face Recognition - Tekan q untuk Keluar', frame)

        # Keluar dari program jika tombol 'q' ditekan
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Bersihkan resource kamera dan tutup jendela
    video_capture.release()
    cv2.destroyAllWindows()
    print("Kamera berhasil dinonaktifkan.")